In [19]:
# Импорты
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow
import json
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score
)
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report
from sklearn import tree
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from catboost import CatBoostClassifier, Pool

In [2]:
# отключаем экспоненциальное отображение чисел в pandas и numpy и делаем удобное форматирование
def smart_float(x):
    if pd.isnull(x):
        return ""
    elif float(x).is_integer():
        return '{:.0f}'.format(x) # отображаем целые числа без нулевой десятичной части
    else:
        return '{:.6f}'.format(x).rstrip('0').rstrip('.') # отображаем числа с плавающей запятой без лишних нулей

pd.set_option('display.float_format', smart_float)
np.set_printoptions(suppress=True)
# Снимаем ограничение на число отображаемых столбцов в pandas
pd.set_option('display.max_columns', None)      # показывать все столбцы
pd.set_option('display.width', None)            # не ограничивать ширину вывода
pd.set_option('display.max_colwidth', None)     # не ограничивать ширину столбца

In [3]:
# Загружаем результаты из прошлого этапа, будем добавлять новые для сравнения
with open('results2.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

In [4]:
# Функция для добавления результатов модели в общий список и отображения итоговой таблицы
def add_result_prob(name, y_true, y_prob, threshold=0.5):

    y_pred = (y_prob >= threshold).astype(int)

    results.append({
        "Model": name,
        "PR-AUC": average_precision_score(y_true, y_prob),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    })
    
def show_results():

    df = pd.DataFrame(results)

    df = df.sort_values("PR-AUC", ascending=False)

    display(df.round(4))

In [5]:
# Функция для оценки модели на разных порогах и сохранения лучших результатов
def evaluate_model_thresholds(
    model=None, 
    X_validate=None, 
    y_validate=None, 
    X_test=None, 
    y_test=None
):
    if model is None:
        model = globals().get('model')
    if X_validate is None:
        X_validate = globals().get('X_validate')
    if y_validate is None:
        y_validate = globals().get('y_validate')
    if X_test is None:
        X_test = globals().get('X_test')
    if y_test is None:
        y_test = globals().get('y_test')

    val_proba = model.predict_proba(X_validate)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]

    val_pr_auc = average_precision_score(y_validate, val_proba)
    test_pr_auc = average_precision_score(y_test, test_proba)

    val_roc_auc = roc_auc_score(y_validate, val_proba)
    test_roc_auc = roc_auc_score(y_test, test_proba)

    print("=== Метрики ===")
    print(f"VALIDATE PR-AUC: {val_pr_auc:.6f}")
    print(f"TEST     PR-AUC: {test_pr_auc:.6f}")
    print(f"VALIDATE ROC-AUC: {val_roc_auc:.6f}")
    print(f"TEST     ROC-AUC: {test_roc_auc:.6f}")

    thresholds = np.linspace(0.01, 0.99, 100)
    rows = []

    for t in thresholds:
        val_pred = (val_proba >= t).astype(int)
        precision = precision_score(y_validate, val_pred, zero_division=0)
        recall = recall_score(y_validate, val_pred, zero_division=0)
        f1 = f1_score(y_validate, val_pred, zero_division=0)
        rows.append({
            'threshold': t,
            'precision': precision,
            'recall': recall,
            'f1': f1
        })

    threshold_df = pd.DataFrame(rows)

    best_f1_row = threshold_df.loc[threshold_df['f1'].idxmax()]
    best_precision_row = threshold_df.loc[threshold_df['precision'].idxmax()]
    best_recall_row = threshold_df.loc[threshold_df['recall'].idxmax()]

    print("\n=== ЛУЧШИЕ ПОРОГИ (VALIDATE) ===")
    print("\nЛучший F1:")
    print(best_f1_row)
    print("\nЛучший Precision:")
    print(best_precision_row)
    print("\nЛучший Recall:")
    print(best_recall_row)

    best_threshold = best_f1_row['threshold']
    test_pred = (test_proba >= best_threshold).astype(int)
    test_precision = precision_score(y_test, test_pred, zero_division=0)
    test_recall = recall_score(y_test, test_pred, zero_division=0)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)

    print("\n=== ТОП-10 ПОРОГОВ ПО F1 (VALIDATE) ===")
    display(threshold_df.sort_values('f1', ascending=False).head(10))

    print("\n=== ФИНАЛЬНЫЕ МЕТРИКИ TEST (лучший порог по F1) ===")
    print(f"Порог: {best_threshold:.4f}")
    print(f"Precision: {test_precision:.6f}")
    print(f"Recall:    {test_recall:.6f}")
    print(f"F1:        {test_f1:.6f}")

    return best_threshold

In [6]:
# Загружаем данные
data = pd.read_parquet('data_tree.parquet')
with open('data_tree_schema.json', 'r', encoding='utf-8') as f:
    schema = json.load(f)

# сначала datetime
for col in schema['datetime_cols']:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors='coerce')

# потом category
for col in schema['category_cols']:
    if col in data.columns:
        data[col] = data[col].astype('category')

# потом остальные типы
for col, dtype_str in schema['dtypes'].items():
    if col not in data.columns:
        continue
    
    if col in schema['datetime_cols'] or col in schema['category_cols']:
        continue
    
    try:
        if dtype_str == 'object':
            data[col] = data[col].astype('string')
        else:
            data[col] = data[col].astype(dtype_str)
    except Exception as e:
        print(f'Не удалось привести {col} к {dtype_str}: {e}')
        
# Делаем MCC категориальным признаком
data['MCC'] = data['MCC'].astype('category')

print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7890638 entries, 0 to 7890637
Data columns (total 55 columns):
 #   Column                               Dtype   
---  ------                               -----   
 0   Amount                               float32 
 1   Use_Chip                             int8    
 2   Is_Online                            int8    
 3   Merchant_State                       category
 4   MCC                                  category
 5   Has_Error                            int8    
 6   Fraud                                int8    
 7   Gender                               int8    
 8   Is_Apartment                         int8    
 9   Total_Debt                           float32 
 10  FICO                                 int16   
 11  Num_Credit_Cards                     int8    
 12  Card_Brand                           category
 13  Card_Type                            category
 14  Has_Chip                             int8    
 15  Cards_Issued   

In [7]:
# Разбиваем на train, validate и test по времени в пропорции 70/15/15
train, test = train_test_split(data, test_size=0.3, shuffle=False)
validate, test = train_test_split(test, test_size=0.5, shuffle=False)
X_train, y_train = train.drop(columns=['Fraud']), train['Fraud']
X_validate, y_validate = validate.drop(columns=['Fraud']), validate['Fraud']
X_test, y_test = test.drop(columns=['Fraud']), test['Fraud']

## 6.1 Бустинг через XGBoost

In [8]:
# Подготовим данные для XGBoost
X_train_xgb = X_train.copy()
X_validate_xgb = X_validate.copy()
X_test_xgb = X_test.copy()

# Задим список категориальных признаков для XGBoost
cat_cols = X_train_xgb.select_dtypes(include=['category']).columns.tolist()

print("Категориальных колонок:", len(cat_cols))
print("Размеры train:", X_train_xgb.shape)
print("Размеры validate:", X_validate_xgb.shape)
print("Размеры test:", X_test_xgb.shape)

# Вычислим веса классов для балансирования при обучении XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("neg:", neg)
print("pos:", pos)
print("scale_pos_weight:", scale_pos_weight)

# Обучим XGBoost с балансировкой классов и гиперпараметрами, подобранными "на глаз"
xgb_model = XGBClassifier(
    n_estimators=700,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    device="cuda",
    enable_categorical=True,
    random_state=42,
    n_jobs=4
)

xgb_model.fit(
    X_train_xgb,
    y_train,
    eval_set=[(X_validate_xgb, y_validate)],
    verbose=50
)


Категориальных колонок: 4
Размеры train: (5523446, 54)
Размеры validate: (1183596, 54)
Размеры test: (1183596, 54)
neg: 5515642
pos: 7804
scale_pos_weight: 706.7711430035879
[0]	validation_0-aucpr:0.15003
[50]	validation_0-aucpr:0.82718
[100]	validation_0-aucpr:0.87173
[150]	validation_0-aucpr:0.88926
[200]	validation_0-aucpr:0.89487
[250]	validation_0-aucpr:0.90050
[300]	validation_0-aucpr:0.90212
[350]	validation_0-aucpr:0.90333
[400]	validation_0-aucpr:0.90478
[450]	validation_0-aucpr:0.90624
[500]	validation_0-aucpr:0.90682
[550]	validation_0-aucpr:0.90685
[600]	validation_0-aucpr:0.90804
[650]	validation_0-aucpr:0.90836
[699]	validation_0-aucpr:0.90865


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,'cuda'
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'aucpr'


In [9]:
print("XGBoost model info")
print("-" * 50)
print("Количество деревьев (n_estimators):", xgb_model.n_estimators)

if hasattr(xgb_model, "best_iteration") and xgb_model.best_iteration is not None:
    print("best_iteration:", xgb_model.best_iteration)

if hasattr(xgb_model, "best_score") and xgb_model.best_score is not None:
    print("best_score:", xgb_model.best_score)

if hasattr(xgb_model, "evals_result"):
    evals_result = xgb_model.evals_result()
    print("\nДоступные наборы в evals_result:", list(evals_result.keys()))
    for dataset_name, metrics in evals_result.items():
        print(f"\n[{dataset_name}]")
        for metric_name, values in metrics.items():
            print(f"  metric: {metric_name}")
            print(f"  last value: {values[-1]:.6f}")
            print(f"  best value: {max(values):.6f}" if "auc" in metric_name.lower() else f"  best/lowest value: {min(values):.6f}")
            print(f"  rounds: {len(values)}")

XGBoost model info
--------------------------------------------------
Количество деревьев (n_estimators): 700

Доступные наборы в evals_result: ['validation_0']

[validation_0]
  metric: aucpr
  last value: 0.908653
  best value: 0.908742
  rounds: 700


In [10]:
best_threshold = evaluate_model_thresholds(model=xgb_model, X_validate=X_validate_xgb, y_validate=y_validate, X_test=X_test_xgb, y_test=y_test)

c:\Users\dmytr\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:774: UserWarning: [23:17:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


=== Метрики ===
VALIDATE PR-AUC: 0.908679
TEST     PR-AUC: 0.860420
VALIDATE ROC-AUC: 0.999393
TEST     ROC-AUC: 0.999512

=== ЛУЧШИЕ ПОРОГИ (VALIDATE) ===

Лучший F1:
threshold   0.297071
precision   0.853179
recall      0.845845
f1          0.849496
Name: 29, dtype: float64

Лучший Precision:
threshold       0.99
precision   0.990164
recall      0.173066
f1          0.294634
Name: 99, dtype: float64

Лучший Recall:
threshold       0.01
precision   0.292716
recall      0.978797
f1           0.45066
Name: 0, dtype: float64

=== ТОП-10 ПОРОГОВ ПО F1 (VALIDATE) ===


,threshold,precision,recall,f1
29,0.297071,0.853179,0.845845,0.849496
28,0.287172,0.848485,0.85043,0.849456
32,0.326768,0.864176,0.834957,0.849315
31,0.316869,0.859659,0.838968,0.849188
30,0.30697,0.855977,0.841261,0.848555
34,0.346566,0.872199,0.825215,0.848057
26,0.267374,0.838384,0.85616,0.847179
33,0.336667,0.866387,0.828653,0.8471
27,0.277273,0.842016,0.852149,0.847052
25,0.257475,0.831672,0.860745,0.845959



=== ФИНАЛЬНЫЕ МЕТРИКИ TEST (лучший порог по F1) ===
Порог: 0.2971
Precision: 0.766703
Recall:    0.795416
F1:        0.780796


In [11]:
add_result_prob(name="XGBoost с простыми гиперпараметрами", y_true=y_test, y_prob=xgb_model.predict_proba(X_test_xgb)[:, 1], threshold=best_threshold)

In [15]:
# Теперь попробуем подобрать гиперпараметры с помощью RandomizedSearchCV
xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    device="cuda",
    enable_categorical=True,
    random_state=42,
    n_jobs=4
)
param_dist = {
    "n_estimators": [300, 500, 700, 1000, 1400],
    "max_depth": [4, 5, 6, 7, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "min_child_weight": [1, 3, 5, 7, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
}
tscv = TimeSeriesSplit(n_splits=3)

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=40,
    scoring="average_precision",
    cv=tscv,
    verbose=10,
    random_state=42,
    n_jobs=1,
    refit=True
)

search.fit(X_train_xgb, y_train)

print("Лучшие параметры:", search.best_params_)
print("Лучший CV PR-AUC:", search.best_score_)

best_xgb = search.best_estimator_

Fitting 3 folds for each of 40 candidates, totalling 120 fits
[CV 1/3; 1/40] START colsample_bytree=0.8, gamma=0.1, learning_rate=0.01, max_depth=5, min_child_weight=7, n_estimators=300, reg_alpha=1.0, reg_lambda=0.5, subsample=1.0
[CV 1/3; 1/40] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.01, max_depth=5, min_child_weight=7, n_estimators=300, reg_alpha=1.0, reg_lambda=0.5, subsample=1.0;, score=0.845 total time=   6.2s
[CV 2/3; 1/40] START colsample_bytree=0.8, gamma=0.1, learning_rate=0.01, max_depth=5, min_child_weight=7, n_estimators=300, reg_alpha=1.0, reg_lambda=0.5, subsample=1.0
[CV 2/3; 1/40] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.01, max_depth=5, min_child_weight=7, n_estimators=300, reg_alpha=1.0, reg_lambda=0.5, subsample=1.0;, score=0.789 total time=   9.9s
[CV 3/3; 1/40] START colsample_bytree=0.8, gamma=0.1, learning_rate=0.01, max_depth=5, min_child_weight=7, n_estimators=300, reg_alpha=1.0, reg_lambda=0.5, subsample=1.0
[CV 3/3; 1/40] END colsamp

In [16]:
# Обучаем лучшую модель
best_xgb.fit(X_train_xgb, y_train, eval_set=[(X_validate_xgb, y_validate)], verbose=50)
print("XGBoost model info")
print("-" * 50)
print("Количество деревьев (n_estimators):", best_xgb.n_estimators)

if hasattr(best_xgb, "best_iteration") and best_xgb.best_iteration is not None:
    print("best_iteration:", best_xgb.best_iteration)

if hasattr(best_xgb, "best_score") and best_xgb.best_score is not None:
    print("best_score:", best_xgb.best_score)

if hasattr(best_xgb, "evals_result"):
    evals_result = best_xgb.evals_result()
    print("\nДоступные наборы в evals_result:", list(evals_result.keys()))
    for dataset_name, metrics in evals_result.items():
        print(f"\n[{dataset_name}]")
        for metric_name, values in metrics.items():
            print(f"  metric: {metric_name}")
            print(f"  last value: {values[-1]:.6f}")
            print(f"  best value: {max(values):.6f}" if "auc" in metric_name.lower() else f"  best/lowest value: {min(values):.6f}")
            print(f"  rounds: {len(values)}")


[0]	validation_0-aucpr:0.12962
[50]	validation_0-aucpr:0.86522
[100]	validation_0-aucpr:0.89257
[150]	validation_0-aucpr:0.90459
[200]	validation_0-aucpr:0.91061
[250]	validation_0-aucpr:0.91315
[300]	validation_0-aucpr:0.91557
[350]	validation_0-aucpr:0.91690
[400]	validation_0-aucpr:0.91735
[450]	validation_0-aucpr:0.91826
[500]	validation_0-aucpr:0.91841
[550]	validation_0-aucpr:0.91848
[600]	validation_0-aucpr:0.91897
[650]	validation_0-aucpr:0.91872
[699]	validation_0-aucpr:0.91895
XGBoost model info
--------------------------------------------------
Количество деревьев (n_estimators): 700

Доступные наборы в evals_result: ['validation_0']

[validation_0]
  metric: aucpr
  last value: 0.918946
  best value: 0.919065
  rounds: 700


In [17]:
best_threshold = evaluate_model_thresholds(model=best_xgb, X_validate=X_validate_xgb, y_validate=y_validate, X_test=X_test_xgb, y_test=y_test)


=== Метрики ===
VALIDATE PR-AUC: 0.918976
TEST     PR-AUC: 0.883040
VALIDATE ROC-AUC: 0.999652
TEST     ROC-AUC: 0.999663

=== ЛУЧШИЕ ПОРОГИ (VALIDATE) ===

Лучший F1:
threshold   0.346566
precision   0.888282
recall      0.838395
f1          0.862618
Name: 34, dtype: float64

Лучший Precision:
threshold   0.980101
precision   0.996212
recall      0.150716
f1          0.261822
Name: 98, dtype: float64

Лучший Recall:
threshold       0.01
precision    0.25646
recall      0.983954
f1          0.406872
Name: 0, dtype: float64

=== ТОП-10 ПОРОГОВ ПО F1 (VALIDATE) ===


,threshold,precision,recall,f1
34,0.346566,0.888282,0.838395,0.862618
33,0.336667,0.882424,0.84298,0.862251
32,0.326768,0.877672,0.846991,0.862059
35,0.356465,0.892091,0.833811,0.861967
28,0.287172,0.85877,0.864183,0.861468
29,0.297071,0.864319,0.85788,0.861087
36,0.366364,0.896402,0.82808,0.860888
37,0.376263,0.900376,0.823496,0.860221
38,0.386162,0.902962,0.821203,0.860144
30,0.30697,0.867055,0.852149,0.859538



=== ФИНАЛЬНЫЕ МЕТРИКИ TEST (лучший порог по F1) ===
Порог: 0.3466
Precision: 0.821531
Recall:    0.797652
F1:        0.809416


In [18]:
add_result_prob(name="XGBoost после подбора гиперпараметров", y_true=y_test, y_prob=best_xgb.predict_proba(X_test_xgb)[:, 1], threshold=best_threshold)
show_results()

,Model,PR-AUC,ROC-AUC,Accuracy,Precision,Recall,F1
24,XGBoost после подбора гиперпараметров,0.883,0.9997,0.9994,0.8215,0.7977,0.8094
23,XGBoost с простыми гиперпараметрами,0.8604,0.9995,0.9993,0.7667,0.7954,0.7808
20,"Random Forest (Ручная оптимизация, тест)",0.7865,0.9993,0.9992,0.7582,0.7149,0.7359
19,"Random Forest (после Optuna, тест)",0.7791,0.9993,0.9991,0.814,0.4891,0.611
18,"Случайный лес (с подобраными гипер-параметрами, test)",0.7609,0.9992,0.9991,0.7221,0.6551,0.687
22,"Стекинг (LogReg + RF, OOF)",0.7199,0.9992,0.9928,0.1713,0.9732,0.2914
17,Базовый RF 500 деревьев (Test),0.718,0.9992,0.999,0.6786,0.6138,0.6446
21,"Random Forest (признаки отобраны через logreg, тест)",0.6388,0.9982,0.9988,0.6044,0.6003,0.6024
11,"Логистическая регрессия (без подбора гиперпараметров) обучена на train+validate, тестирование на test",0.5957,0.9963,0.9989,0.7781,0.398,0.5266
9,Логистическая регрессия (без подбора гиперпараметров) на тесте,0.5957,0.9963,0.9989,0.7781,0.398,0.5266


На данный момент мы видим наилучший результат именно у этой модели. 80% случаев мошенничества верно определяются моделью, при этом 82% всех алёртов оказываются истинныыми. Это позволяет использовать модель даже для автоматических блокировок, т.к. случаев жалоб клиентов будет достаточно мало, чтобы отдел мониторинга мог оперативно их рассматривать.

## 6.2 Бустинг через CatBoost

Несмотря на огромный успех XGBoosting мы попробуем также обучить модель на CatBoost. Дело в том, что именно этот пакет построен специально для работу с большими табличными категориальными данными. В данном случае категории очень важны для нашего проекта, по этому есть смысл проверить его возможности.

In [29]:
# Делаем копии для CatBoost, чтобы не испортить данные для XGBoost
X_train_cb = X_train.copy()
X_validate_cb = X_validate.copy()
X_test_cb = X_test.copy()

# Выделим категориальные признаки для CatBoost
cat_cols = X_train_cb.select_dtypes(include=['category', 'object', 'string']).columns.tolist()

# CatBoost потребовал тип данных string для категорий
for col in cat_cols:
    X_train_cb[col] = X_train_cb[col].astype("string").fillna("__MISSING__")
    X_validate_cb[col] = X_validate_cb[col].astype("string").fillna("__MISSING__")
    X_test_cb[col] = X_test_cb[col].astype("string").fillna("__MISSING__")

print("Категориальных колонок:", len(cat_cols))
print("Train shape:", X_train_cb.shape)
print("Validate shape:", X_validate_cb.shape)
print("Test shape:", X_test_cb.shape)

train_pool = Pool(
    data=X_train_cb,
    label=y_train,
    cat_features=cat_cols
)

validate_pool = Pool(
    data=X_validate_cb,
    label=y_validate,
    cat_features=cat_cols
)

test_pool = Pool(
    data=X_test_cb,
    label=y_test,
    cat_features=cat_cols
)

# Вычислим веса классов для балансирования при обучении CatBoost
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / pos

print("neg:", neg)
print("pos:", pos)
print("scale_pos_weight:", scale_pos_weight)

# Обучим CatBoost с балансировкой классов и гиперпараметрами, подобранными "на глаз"
cat_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss", # CatBoost не может считать PRAUC в GPU, а без GPU слишком долго
    eval_metric="Logloss",
    task_type="GPU",
    devices="0",
    random_seed=42,
    verbose=100,
    scale_pos_weight=scale_pos_weight
)

cat_model.fit(
    train_pool,
    eval_set=validate_pool,
    use_best_model=True
)

Категориальных колонок: 4
Train shape: (5523446, 54)
Validate shape: (1183596, 54)
Test shape: (1183596, 54)
neg: 5515642
pos: 7804
scale_pos_weight: 706.7711430035879
0:	learn: 0.5746208	test: 0.5621575	best: 0.5621575 (0)	total: 323ms	remaining: 10m 45s
100:	learn: 0.0427953	test: 0.0421348	best: 0.0421348 (100)	total: 26.9s	remaining: 8m 25s
200:	learn: 0.0286565	test: 0.0397652	best: 0.0384469 (190)	total: 53.5s	remaining: 7m 58s
300:	learn: 0.0211241	test: 0.0414152	best: 0.0384469 (190)	total: 1m 20s	remaining: 7m 32s
400:	learn: 0.0158202	test: 0.0446055	best: 0.0384469 (190)	total: 1m 46s	remaining: 7m 6s
500:	learn: 0.0121949	test: 0.0482653	best: 0.0384469 (190)	total: 2m 13s	remaining: 6m 39s
600:	learn: 0.0098489	test: 0.0558717	best: 0.0384469 (190)	total: 2m 40s	remaining: 6m 13s
700:	learn: 0.0080950	test: 0.0606224	best: 0.0384469 (190)	total: 3m 7s	remaining: 5m 46s
800:	learn: 0.0068923	test: 0.0674297	best: 0.0384469 (190)	total: 3m 33s	remaining: 5m 19s
900:	learn: 

По логам обучения CatBoost видно, что модель достигает наилучшего качества на валидационной выборке уже примерно на 190-й итерации. Дальнейшее увеличение числа итераций приводит к ухудшению качества на валидации при продолжающемся улучшении на обучающей выборке, что указывает на переобучение. Поэтому при текущих гиперпараметрах оптимальная длина бустинга составляет около 190–200 итераций, а дальнейшее обучение не даёт пользы.

In [30]:
# Получим предсказания вероятностей для validate и test
validate_proba_cb = cat_model.predict_proba(validate_pool)[:, 1]
test_proba_cb = cat_model.predict_proba(test_pool)[:, 1]

cat_results = pd.DataFrame([
    {
        "Dataset": "validate",
        "PR-AUC": average_precision_score(y_validate, validate_proba_cb),
        "ROC-AUC": roc_auc_score(y_validate, validate_proba_cb),
    },
    {
        "Dataset": "test",
        "PR-AUC": average_precision_score(y_test, test_proba_cb),
        "ROC-AUC": roc_auc_score(y_test, test_proba_cb),
    }
])

cat_results

,Dataset,PR-AUC,ROC-AUC
0,validate,0.844094,0.998926
1,test,0.828129,0.999396


In [31]:
best_threshold = evaluate_model_thresholds(model=cat_model, X_validate=X_validate_cb, y_validate=y_validate, X_test=X_test_cb, y_test=y_test)

=== Метрики ===
VALIDATE PR-AUC: 0.844094
TEST     PR-AUC: 0.828129
VALIDATE ROC-AUC: 0.998926
TEST     ROC-AUC: 0.999396

=== ЛУЧШИЕ ПОРОГИ (VALIDATE) ===

Лучший F1:
threshold   0.980101
precision   0.746261
recall      0.800573
f1          0.772463
Name: 98, dtype: float64

Лучший Precision:
threshold       0.99
precision   0.848722
recall      0.684814
f1          0.758008
Name: 99, dtype: float64

Лучший Recall:
threshold       0.01
precision   0.008666
recall      0.999427
f1          0.017182
Name: 0, dtype: float64

=== ТОП-10 ПОРОГОВ ПО F1 (VALIDATE) ===


,threshold,precision,recall,f1
98,0.980101,0.746261,0.800573,0.772463
99,0.99,0.848722,0.684814,0.758008
97,0.970202,0.663805,0.841834,0.742294
96,0.960303,0.602358,0.87851,0.714685
95,0.950404,0.547077,0.895702,0.67927
94,0.940505,0.503,0.912894,0.648616
93,0.930606,0.464131,0.923209,0.617715
92,0.920707,0.429252,0.930086,0.587405
91,0.910808,0.399172,0.939255,0.560246
90,0.900909,0.373838,0.944986,0.535737



=== ФИНАЛЬНЫЕ МЕТРИКИ TEST (лучший порог по F1) ===
Порог: 0.9801
Precision: 0.719008
Recall:    0.778088
F1:        0.747383


In [32]:
add_result_prob(name="CatBoost без подбора параметров", y_true=y_test, y_prob=cat_model.predict_proba(test_pool)[:, 1], threshold=best_threshold)
show_results()

,Model,PR-AUC,ROC-AUC,Accuracy,Precision,Recall,F1
24,XGBoost после подбора гиперпараметров,0.883,0.9997,0.9994,0.8215,0.7977,0.8094
23,XGBoost с простыми гиперпараметрами,0.8604,0.9995,0.9993,0.7667,0.7954,0.7808
25,CatBoost без подбора параметров,0.8281,0.9994,0.9992,0.719,0.7781,0.7474
20,"Random Forest (Ручная оптимизация, тест)",0.7865,0.9993,0.9992,0.7582,0.7149,0.7359
19,"Random Forest (после Optuna, тест)",0.7791,0.9993,0.9991,0.814,0.4891,0.611
18,"Случайный лес (с подобраными гипер-параметрами, test)",0.7609,0.9992,0.9991,0.7221,0.6551,0.687
22,"Стекинг (LogReg + RF, OOF)",0.7199,0.9992,0.9928,0.1713,0.9732,0.2914
17,Базовый RF 500 деревьев (Test),0.718,0.9992,0.999,0.6786,0.6138,0.6446
21,"Random Forest (признаки отобраны через logreg, тест)",0.6388,0.9982,0.9988,0.6044,0.6003,0.6024
9,Логистическая регрессия (без подбора гиперпараметров) на тесте,0.5957,0.9963,0.9989,0.7781,0.398,0.5266


In [34]:
# Попробуем подобрать гиперпараметры для CatBoost с помощью RandomizedSearchCV
tscv = TimeSeriesSplit(n_splits=3)
cb_base = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="Logloss",
    task_type="GPU",
    devices="0",
    random_seed=42,
    verbose=False,
    cat_features=cat_cols
)
param_dist_cb = {
    "iterations": [300, 500, 700, 1000, 1500, 2000],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.07],
    "depth": [4, 5, 6, 7, 8, 10],
    "l2_leaf_reg": [1, 3, 5, 7, 9, 12],
    "border_count": [128, 254],
    "bagging_temperature": [0, 0.5, 1, 2, 5],
    "random_strength": [0.5, 1, 2, 5],
    "scale_pos_weight": [scale_pos_weight * k for k in [0.75, 1.0, 1.25, 1.5]]
}
search_cb = RandomizedSearchCV(
    estimator=cb_base,
    param_distributions=param_dist_cb,
    n_iter=30,
    scoring="average_precision",
    cv=tscv,
    refit=True,
    verbose=2,
    random_state=42,
    n_jobs=1,
    return_train_score=False,
    error_score="raise"
)
search_cb.fit(X_train_cb, y_train)

print("Лучшие параметры:")
print(search_cb.best_params_)

print("\nЛучший CV score (PR-AUC):")
print(search_cb.best_score_)

best_cat_model = search_cb.best_estimator_

Fitting 3 folds for each of 30 candidates, totalling 90 fits
[CV] END bagging_temperature=2, border_count=254, depth=4, iterations=700, l2_leaf_reg=1, learning_rate=0.03, random_strength=1, scale_pos_weight=883.4639287544849; total time=  28.2s
[CV] END bagging_temperature=2, border_count=254, depth=4, iterations=700, l2_leaf_reg=1, learning_rate=0.03, random_strength=1, scale_pos_weight=883.4639287544849; total time=  50.7s
[CV] END bagging_temperature=2, border_count=254, depth=4, iterations=700, l2_leaf_reg=1, learning_rate=0.03, random_strength=1, scale_pos_weight=883.4639287544849; total time= 1.3min
[CV] END bagging_temperature=5, border_count=128, depth=6, iterations=2000, l2_leaf_reg=12, learning_rate=0.07, random_strength=0.5, scale_pos_weight=1060.1567145053818; total time= 1.6min
[CV] END bagging_temperature=5, border_count=128, depth=6, iterations=2000, l2_leaf_reg=12, learning_rate=0.07, random_strength=0.5, scale_pos_weight=1060.1567145053818; total time= 2.9min
[CV] END 

In [35]:
validate_proba_cb = best_cat_model.predict_proba(X_validate_cb)[:, 1]
test_proba_cb = best_cat_model.predict_proba(X_test_cb)[:, 1]

cat_results = pd.DataFrame([
    {
        "Dataset": "validate",
        "PR-AUC": average_precision_score(y_validate, validate_proba_cb),
        "ROC-AUC": roc_auc_score(y_validate, validate_proba_cb)
    },
    {
        "Dataset": "test",
        "PR-AUC": average_precision_score(y_test, test_proba_cb),
        "ROC-AUC": roc_auc_score(y_test, test_proba_cb)
    }
])

cat_results

,Dataset,PR-AUC,ROC-AUC
0,validate,0.892846,0.999137
1,test,0.838488,0.998942


In [36]:
best_threshold = evaluate_model_thresholds(model=best_cat_model, X_validate=X_validate_cb, y_validate=y_validate, X_test=X_test_cb, y_test=y_test)

=== Метрики ===
VALIDATE PR-AUC: 0.892846
TEST     PR-AUC: 0.838488
VALIDATE ROC-AUC: 0.999137
TEST     ROC-AUC: 0.998942

=== ЛУЧШИЕ ПОРОГИ (VALIDATE) ===

Лучший F1:
threshold   0.702929
precision    0.87069
recall      0.810315
f1          0.839418
Name: 70, dtype: float64

Лучший Precision:
threshold       0.99
precision   0.966052
recall      0.570774
f1          0.717579
Name: 99, dtype: float64

Лучший Recall:
threshold       0.01
precision   0.283057
recall      0.967908
f1          0.438019
Name: 0, dtype: float64

=== ТОП-10 ПОРОГОВ ПО F1 (VALIDATE) ===


,threshold,precision,recall,f1
70,0.702929,0.87069,0.810315,0.839418
68,0.683131,0.863554,0.816046,0.839128
69,0.69303,0.865772,0.813181,0.838652
67,0.673232,0.859988,0.816619,0.837743
72,0.722727,0.874532,0.802865,0.837168
73,0.732626,0.87594,0.801146,0.836875
71,0.712828,0.871508,0.804585,0.83671
66,0.663333,0.855516,0.817765,0.836214
65,0.653434,0.852205,0.819484,0.835524
64,0.643535,0.849437,0.821203,0.835082



=== ФИНАЛЬНЫЕ МЕТРИКИ TEST (лучший порог по F1) ===
Порог: 0.7029
Precision: 0.779370
Recall:    0.760201
F1:        0.769666


In [37]:
add_result_prob(name="CatBoost после подбора гиперпараметров", y_true=y_test, y_prob=best_cat_model.predict_proba(X_test_cb)[:, 1], threshold=best_threshold)
show_results()

,Model,PR-AUC,ROC-AUC,Accuracy,Precision,Recall,F1
24,XGBoost после подбора гиперпараметров,0.883,0.9997,0.9994,0.8215,0.7977,0.8094
23,XGBoost с простыми гиперпараметрами,0.8604,0.9995,0.9993,0.7667,0.7954,0.7808
26,CatBoost после подбора гиперпараметров,0.8385,0.9989,0.9993,0.7794,0.7602,0.7697
25,CatBoost без подбора параметров,0.8281,0.9994,0.9992,0.719,0.7781,0.7474
20,"Random Forest (Ручная оптимизация, тест)",0.7865,0.9993,0.9992,0.7582,0.7149,0.7359
19,"Random Forest (после Optuna, тест)",0.7791,0.9993,0.9991,0.814,0.4891,0.611
18,"Случайный лес (с подобраными гипер-параметрами, test)",0.7609,0.9992,0.9991,0.7221,0.6551,0.687
22,"Стекинг (LogReg + RF, OOF)",0.7199,0.9992,0.9928,0.1713,0.9732,0.2914
17,Базовый RF 500 деревьев (Test),0.718,0.9992,0.999,0.6786,0.6138,0.6446
21,"Random Forest (признаки отобраны через logreg, тест)",0.6388,0.9982,0.9988,0.6044,0.6003,0.6024


Как видим, XGBoost показал немного лучший результат, чем CatBoost. Сохраним модель XGBoost в виде .pkl файла

In [40]:
import pickle
# Создаем .pkl для best_xgb
with open('best_xgb.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)

In [43]:
X_test.to_pickle("prod_testing.pkl")

In [41]:
# Сохраним список результатов
with open('results2.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

## Общий вывод

Поиск мошеннических транзакций является задачей классификации с ярко выраженным дисбалансом классов. Основной трудностью подготовки данных является создание достаточного числа признаков, не имеющих линейной зависимости между собой, а также удаление признаков, которые излишне коррелируются с целевой переменной, т.к. это может привести к не универсальности модели.
Кроме того, глубокие базы данных за большие периоды времени бесполезны для построения моделей из-за смены паттерном мошенничества с течением времени, а также сменой эпох в банкинге. Т.е. данные за 1990-2010 годы только мешают предсказанию из-за существенной смены технологий и поведения пользователей.
В качестве baseline была взята не-ML модель на базе скоринговых правил (были взяты реальные правила одного из существующих банков). 
Даже самая слабая модель линейной регрессии показала улучшение качества распознования.
Классы являеются линейно разделимыми, однако из-за очень большого числа слабых сигналов линейная регрессия ограничена в своей эффективности. Деревья решений и их ансамблевые модели показывают отличный результат, существенно превосходящий линейную регрессию. Дальнейшее улучшение качества модели показало, что наиболее эффективной будет модель на базе градиентного бустинга XGBoost. 
Стекинг показал слабую эффективность в силу слишком большшого различия качества моделей. Фактически, для хорошего стекинга было бы необходимо создавать несколько умышленно слабых моделей, что в условиях задачи лишено смысла.
Берём модель градиентного бустинга для демонстрации возможностей продакшн. 